In [5]:
import tensorflow_hub as hub
import joblib
import gzip
import kipoiseq
from kipoiseq import Interval
import pyfaidx
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "MIG-f80e9374-504a-571b-bac0-6fb00750db4c" 
os.environ["WORLD_SIZE"] = "0"

%matplotlib inline
%config InlineBackend.figure_format = 'retina'

In [6]:
transform_path = 'gs://dm-enformer/models/enformer.finetuned.SAD.robustscaler-PCA500-robustscaler.transform.pkl'
model_path = 'https://tfhub.dev/deepmind/enformer/1'
#fasta_file = '/mnt/lab_data2/anusri/chrombpnet/reference/hg38.genome.fa'
#clinvar_vcf = 'clinvar.vcf.gz'

# Download targets from Basenji2 dataset 
# Cite: Kelley et al Cross-species regulatory sequence activity prediction. PLoS Comput. Biol. 16, e1008050 (2020).
targets_txt = 'https://raw.githubusercontent.com/calico/basenji/master/manuscripts/cross2020/targets_human.txt'
df_targets = pd.read_csv(targets_txt, sep='\t')
df_targets.head(3)

"""### Code (double click on the title to show the code)"""

# @title `Enformer`, `EnformerScoreVariantsNormalized`, `EnformerScoreVariantsPCANormalized`,
SEQUENCE_LENGTH = 393216

In [7]:
import tensorflow as tf
class Enformer:

  def __init__(self, tfhub_url):
    self._model = hub.load(tfhub_url).model

  def predict_on_batch(self, inputs):
    predictions = self._model.predict_on_batch(inputs)
    return {k: v.numpy() for k, v in predictions.items()}

  @tf.function
  def contribution_input_grad(self, input_sequence,
                              target_mask, output_head='human'):
    input_sequence = input_sequence[tf.newaxis]

    target_mask_mass = tf.reduce_sum(target_mask)
    with tf.GradientTape() as tape:
      tape.watch(input_sequence)
      prediction = tf.reduce_sum(
          target_mask[tf.newaxis] *
          self._model.predict_on_batch(input_sequence)[output_head]) / target_mask_mass

    input_grad = tape.gradient(prediction, input_sequence) * input_sequence
    input_grad = tf.squeeze(input_grad, axis=0)
    return tf.reduce_sum(input_grad, axis=-1)



In [13]:
model = Enformer(model_path)
import pyfaidx
from tangermeme.utils import one_hot_encode
from tangermeme.utils import random_one_hot
from tangermeme.ersatz import substitute
import one_hot

SEQLEN=196608*2

# pos0=5271075
# allele='G'
# altallele='A'
# chrm = 'chr11'

pos0=5271188
chrm = 'chr11'

# pos0=155271607
# allele='A'
# altallele='C'
# chrm = 'chr1'

# pos0=5271228
# allele='G'
# altallele='A'
# chrm = 'chr11'

# pos0=5271230
# allele='G'
# altallele='A'
# chrm = 'chr11'

genome = pyfaidx.Fasta("/mnt/lab_data2/anusri/chrombpnet/reference/male.hg19.fa")
#refseq = genome[chrm][pos0-SEQLEN//2:pos0].seq.upper() + allele + genome[chrm][pos0+1:pos0+SEQLEN//2].seq.upper()
#altseq = genome[chrm][pos0-SEQLEN//2:pos0].seq.upper() + altallele + genome[chrm][pos0+1:pos0+SEQLEN//2].seq.upper()

refseq = genome[chrm][pos0-SEQLEN//2:pos0+SEQLEN//2].seq.upper() 
altseq = genome[chrm][pos0-SEQLEN//2:pos0+SEQLEN//2].seq.upper() 


refX = one_hot.dna_to_one_hot(refseq)
refX.shape

altX = one_hot.dna_to_one_hot(altseq)
altX.shape

ResourceExhaustedError: /tmp/tfhub_modules/c444fdff3e183daf686869692c26e00391f6773c.d5a6359ed7984b6e81d9206b3ca0a30f.tmp; No space left on device

In [11]:
refpred = model.predict_on_batch(refX.reshape((-1,SEQLEN,4)))['human'][0]
altpred = model.predict_on_batch(altX.reshape((-1,SEQLEN,4)))['human'][0]

NameError: name 'model' is not defined

In [ ]:
plt.plot(refpred[896//2-4:896//2+4,121], label="ref")
plt.plot(altpred[896//2-4:896//2+4,121], label="alt")
plt.legend()

In [ ]:
target_mask = np.zeros_like(refpred)
for idx in range(896//2-4,896//2+4):
  #print(idx)
  target_mask[idx, 121] = 1

refcontrib = model.contribution_input_grad(refX.reshape((SEQLEN,4)).astype(np.float32),target_mask)
altcontrib = model.contribution_input_grad(altX.reshape((SEQLEN,4)).astype(np.float32),target_mask)

refcontribs = np.transpose(refX.squeeze())[None,:,:] * refcontrib[None,None,:]
altcontribs = np.transpose(altX.squeeze())[None,:,:] * altcontrib[None,None,:]
#print(refcontrib)
max_v1 = tf.reduce_max(tf.concat([refcontrib,altcontrib], axis=-1) )


In [ ]:
import matplotlib.pyplot as plt
import seaborn; seaborn.set_style('white')
from tangermeme.plot import plot_logo
import torch 
# import tensorflow.python.ops.numpy_ops.np_config as np_config
# np_config.enable_numpy_behavior()


fig, axs = plt.subplots(nrows=2, ncols=1, figsize=(12, 4)) # Creates 3 vertically stacked subplots

plot_logo(refcontribs[0,:,393216//2-150:393216//2+150].numpy(), ax=axs[0])
plot_logo(altcontribs[0,:,393216//2-150:393216//2+150].numpy(), ax=axs[1])
axs[0].set_title("Ref Allele", fontsize=8)
axs[1].set_title("Alt Allele", fontsize=8)

axs[1].set_xlabel("Genomic Coordinate")
axs[1].set_ylabel("Gradients", fontsize=6)
# plt.title("Ref Allele")
axs[0].set_ylim((0,0.2))
axs[1].set_ylim((0,0.2))
plt.tight_layout()


#title="enformer_"+chrm+"_"+str(pos0)+"_"+allele+"_"+altallele
#plt.savefig("subfigs/"+title+".pdf", transparent=True, dpi=300)
